In [ ]:
import tensorflow as tf
import numpy as np

# 0 = Dark, 1 = Normal, 2 = Bright
x = np.array([
    5, 10, 15, 20, 25, 30,
    40, 45, 50, 55, 60,
    70, 75, 80, 85, 90, 95
], dtype=np.float32)

y = np.array([
    0, 0, 0, 0, 0, 0,
    1, 1, 1, 1, 1,
    2, 2, 2, 2, 2, 2
], dtype=np.int32)

x_scaled = x / 100.0

model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(x_scaled, y, epochs=500, verbose=0)

print("Model training completed!")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("light_classifier.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully!")

Model training completed!
Saved artifact at '/tmp/tmpkbbq7v44'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor_11')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  138603134860560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138603134859792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138603134860368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138603134859984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138603134860944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138603134860752: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved successfully!


In [ ]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(
    model_path="light_classifier.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

test = np.array([10, 50, 90], dtype=np.float32)

classes = ["Dark", "Normal", "Bright"]

for value in test:
    input_data = np.array([[value / 100.0]], dtype=np.float32)

    interpreter.set_tensor(
        input_details[0]["index"],
        input_data
    )

    interpreter.invoke()

    output = interpreter.get_tensor(
        output_details[0]["index"]
    )

    predicted_class = np.argmax(output)

    print("Light Level =", value, "->", classes[predicted_class])

Light Level = 10.0 -> Dark
Light Level = 50.0 -> Normal
Light Level = 90.0 -> Bright


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
